# **Verdadera Democracia con Algoritmos Genéticos**

Queremos distribuir 50 entidades del Estado (con pesos políticos) entre 5 partidos, de forma que el poder recibido por cada partido sea proporcional a sus curules en el Congreso.

## Definimos el Problema

Aquí los elementos clave de nuestro Algoritmo Genético:

-   **Individuo (cromosoma)**: Un arreglo de 50 genes, donde cada gen es un número del 0 al 4 (representando el partido asignado a esa entidad).
-   **Gen**: Partido político que recibe la entidad `i`.
-   **Población**: Conjunto de `N` cromosomas candidatos.
-   **Aptitud (fitness)**: Mide qué tan bien se distribuye el poder proporcional a las curules.

## Generar los Datos Iniciales

1.  **Distribuir 50 curules entre 5 partidos** (distribución NO uniforme):
    ```
    P0=18, P1=12, P2=9, P3=7, P4=4 → suma=50
    ```

2.  **Asignar aleatoriamente un peso político** `[1..100]` a cada entidad:
    ```
    entidad[0]=87, entidad[1]=34, entidad[2]=61, ...
    ```

3.  **Calcular proporción ideal de poder por partido**:
    ```python
    proporcion[i] = curules[i] / 50
    poder_ideal[i] = proporcion[i] × suma_total_poder
    ```

### Codificación del Cromosoma

Un cromosoma es una secuencia de asignaciones de entidades a partidos:

```
Cromosoma = [2, 0, 1, 4, 0, 3, 1, 2, ... ] (longitud 50)
             ↑                               
          Entidad 0 → asignada al Partido 2
```

### Función de Aptitud (Fitness)

Esta es la pieza clave del AG. Para un cromosoma dado:

1.  Calcular el poder obtenido por cada partido:
    `poder_obtenido[p] = Σ peso[i]` para toda entidad `i` asignada al partido `p`

2.  Calcular el error de distribución:
    `error = Σ |poder_obtenido[p] - poder_ideal[p]|` para `p` en `[0..4]`

3.  Calcular la aptitud (fitness):
    `fitness = 1 / (1 + error)`
    *   Un valor **más alto** indica una mejor distribución.
    *   `fitness = 1` significa una distribución perfecta.

### Crear Población Inicial

Para cada individuo en la población (ej. 200 individuos):

-   Generar 50 genes aleatorios en `[0..4]`:
    ```python
    cromosoma = [random(0,4) for _ in range(50)]
    ```

### Selección (Selection)

Usamos la **Selección por Torneo**:

1.  Elegir aleatoriamente `k` individuos (ej. `k=5`).
2.  El individuo con mayor fitness

In [2]:
import random
import numpy as np
from copy import deepcopy

In [17]:
# ─────────────────────────────────────────────────────
# 1. SEMILLA (reproducibilidad opcional)
# ─────────────────────────────────────────────────────
SEED = 33
random.seed(SEED)
np.random.seed(SEED)

In [4]:
# ─────────────────────────────────────────────────────
# 2. DATOS DEL PROBLEMA
# ─────────────────────────────────────────────────────

PARTIDOS = ["Partido A", "Partido B", "Partido C", "Partido D", "Partido E"]
N_PARTIDOS  = 5
N_ENTIDADES = 50
N_CURULES   = 50

#Nombres de entidades generadas por Claude.AI
ENTIDADES = [
    "Min. Hacienda",        "Min. Interior",       "Min. Defensa",
    "Min. Relaciones Ext.", "Min. Justicia",        "Min. Educación",
    "Min. Salud",           "Min. Trabajo",         "Min. Comercio",
    "Min. Ambiente",        "Min. Transporte",      "Min. TIC",
    "Min. Cultura",         "Min. Ciencia",         "Min. Agricultura",
    "Min. Minas",           "Min. Vivienda",        "Min. Deporte",
    "DNP",                  "DIAN",                 "Banco de la República",
    "Contraloría",          "Procuraduría",         "Fiscalía",
    "Defensoría del Pueblo","Consejo de Estado",    "Corte Constitucional",
    "Corte Suprema",        "CSJ",                  "DANE",
    "ICBF",                 "SENA",                 "Invías",
    "ANI",                  "ANH",                  "ANLA",
    "CAR Nacional",         "Unidad Víctimas",      "UNGRD",
    "Cancillería Consular", "Agencia Espacial",     "Colciencias",
    "Icontec",              "Super. Financiera",    "SIC",
    "Super. Salud",         "IGAC",                 "SGR",
    "Fonpet",               "CNTV",
]


def generar_datos_aleatorios():
    """
    Genera una distribución NO uniforme de curules y
    pesos políticos aleatorios [1..100] para cada entidad.
    """
    cortes = sorted(random.sample(range(2, N_CURULES - N_PARTIDOS + 1),
                                  N_PARTIDOS - 1))
    limites = [0] + cortes + [N_CURULES]
    curules = [limites[i+1] - limites[i] for i in range(N_PARTIDOS)]
    random.shuffle(curules)

    pesos = [random.randint(1, 100) for _ in range(N_ENTIDADES)]
    return curules, pesos

In [6]:
# ─────────────────────────────────────────────────────
# 3. FUNCIONES AUXILIARES
# ─────────────────────────────────────────────────────

def poder_ideal(curules: list, pesos: list) -> list:
    """Calcula cuánto poder debería recibir cada partido según sus curules."""
    total_poder = sum(pesos)
    return [c / N_CURULES * total_poder for c in curules]


def poder_obtenido(cromosoma: list, pesos: list) -> list:
    """Suma el peso de las entidades asignadas a cada partido."""
    po = [0.0] * N_PARTIDOS
    for i, partido in enumerate(cromosoma):
        po[partido] += pesos[i]
    return po

In [8]:
# ─────────────────────────────────────────────────────
# 4. FUNCIÓN DE APTITUD (FITNESS)
# ─────────────────────────────────────────────────────

def fitness(cromosoma: list, curules: list, pesos: list) -> float:
    """
    Mide qué tan bien distribuido está el poder.

    fitness = 1 / (1 + error_total)
      → 1.0  = distribución perfecta
      → ~0   = muy desigual
    """
    po    = poder_obtenido(cromosoma, pesos)
    ideal = poder_ideal(curules, pesos)
    error = sum(abs(po[p] - ideal[p]) for p in range(N_PARTIDOS))
    return 1.0 / (1.0 + error)

In [10]:
# ─────────────────────────────────────────────────────
# 5. OPERADORES GENÉTICOS
# ─────────────────────────────────────────────────────

def cromosoma_aleatorio() -> list:
    """Crea un individuo aleatorio (asignación de partidos a entidades)."""
    return [random.randint(0, N_PARTIDOS - 1) for _ in range(N_ENTIDADES)]


def seleccion_torneo(poblacion: list, fitnesses: list, k: int = 5) -> list:
    """Selecciona al mejor individuo de k candidatos aleatorios."""
    candidatos = random.sample(range(len(poblacion)), k)
    ganador = max(candidatos, key=lambda i: fitnesses[i])
    return poblacion[ganador][:]


def cruce_un_punto(padre1: list, padre2: list) -> tuple:
    """Cruce en un punto: genera dos hijos."""
    punto = random.randint(1, N_ENTIDADES - 1)
    hijo1 = padre1[:punto] + padre2[punto:]
    hijo2 = padre2[:punto] + padre1[punto:]
    return hijo1, hijo2


def mutacion(cromosoma: list, prob_mut: float) -> list:
    """Con probabilidad prob_mut, reasigna aleatoriamente cada gen."""
    return [
        random.randint(0, N_PARTIDOS - 1) if random.random() < prob_mut else gen
        for gen in cromosoma
    ]

In [28]:
# ─────────────────────────────────────────────────────
# 6. ALGORITMO GENÉTICO PRINCIPAL
# ─────────────────────────────────────────────────────

def algoritmo_genetico(
    curules: list,
    pesos: list,
    tam_poblacion: int  = 200,
    n_generaciones: int = 1000,
    prob_cruce: float   = 0.85,
    prob_mutacion: float= 0.02,
    objetivo_fitness: float = 0.98,
    verbose: bool       = True,
) -> tuple:
    """
    Ejecuta el AG y devuelve (mejor_cromosoma, mejor_fitness, historial).

    Parámetros
    ----------
    curules         : lista con curules por partido (suma = 50)
    pesos           : lista con peso político de cada entidad (1-100)
    tam_poblacion   : número de individuos en la población
    n_generaciones  : máximo de generaciones
    prob_cruce      : probabilidad de cruce entre dos padres
    prob_mutacion   : probabilidad de mutar cada gen
    objetivo_fitness: umbral para parada temprana
    verbose         : imprime progreso cada 100 generaciones

    Retorna
    -------
    mejor_cromosoma : lista de 50 enteros (0-4)
    mejor_fitness   : float
    historial       : lista de mejores fitness por generación
    """

    # --- Inicialización ---
    poblacion = [cromosoma_aleatorio() for _ in range(tam_poblacion)]
    historial = []
    mejor_global = None
    mejor_fit_global = -1

    for gen in range(n_generaciones):

        # Evaluar toda la población
        fits = [fitness(c, curules, pesos) for c in poblacion]

        # Identificar el mejor de esta generación
        idx_mejor = max(range(tam_poblacion), key=lambda i: fits[i])
        if fits[idx_mejor] > mejor_fit_global:
            mejor_fit_global = fits[idx_mejor]
            mejor_global = poblacion[idx_mejor][:]

        historial.append(mejor_fit_global)

        if verbose and (gen % 100 == 0 or mejor_fit_global >= objetivo_fitness):
            print(f"  Gen {gen:>4} | fitness = {mejor_fit_global:.6f}")

        # Criterio de parada temprana
        if mejor_fit_global >= objetivo_fitness:
            if verbose:
                print(f"\n  ✓ Convergió en la generación {gen}")
            break

        # --- Nueva generación ---
        nueva_poblacion = [mejor_global[:]]  # Elitismo: conservar al mejor

        while len(nueva_poblacion) < tam_poblacion:
            # Selección por torneo
            p1 = seleccion_torneo(poblacion, fits)
            p2 = seleccion_torneo(poblacion, fits)

            # Cruce
            if random.random() < prob_cruce:
                h1, h2 = cruce_un_punto(p1, p2)
            else:
                h1, h2 = p1[:], p2[:]

            # Mutación
            h1 = mutacion(h1, prob_mutacion)
            h2 = mutacion(h2, prob_mutacion)

            nueva_poblacion.extend([h1, h2])

        poblacion = nueva_poblacion[:tam_poblacion]

    return mejor_global, mejor_fit_global, historial

In [33]:
# ─────────────────────────────────────────────────────
# 7. MOSTRAR RESULTADOS
# ─────────────────────────────────────────────────────

def mostrar_resultados(cromosoma: list, curules: list, pesos: list, fit: float):
    """Imprime la matriz de poder resultante."""
    ideal = poder_ideal(curules, pesos)
    po    = poder_obtenido(cromosoma, pesos)
    total = sum(pesos)

    print("\n" + "="*62)
    print("  MATRIZ DE PODER — RESULTADO FINAL")
    print("="*62)
    print(f"  Fitness alcanzado : {fit:.6f}")
    print(f"  Poder total       : {total} puntos")
    print()

    # Resumen por partido
    print(f"  {'Partido':<14} {'Curules':>7} {'Ideal':>8} {'Obtenido':>10} {'Error':>8}")
    print("  " + "-"*50)
    for p in range(N_PARTIDOS):
        error = po[p] - ideal[p]
        signo = "+" if error >= 0 else ""
        print(f"  {PARTIDOS[p]:<14} {curules[p]:>7} {ideal[p]:>8.1f} "
              f"{po[p]:>10.1f} {signo}{error:>7.1f}")
    print()

    # Entidades asignadas a cada partido
    print("  ENTIDADES POR PARTIDO:")
    print("  " + "-"*50)
    for p in range(N_PARTIDOS):
        asignadas = [(ENTIDADES[i], pesos[i])
                     for i in range(N_ENTIDADES) if cromosoma[i] == p]
        asignadas.sort(key=lambda x: -x[1])  # ordenar por peso desc
        nombres = ", ".join(f"{n} ({w})" for n, w in asignadas)
        print(f"\n  {PARTIDOS[p]} [{curules[p]} cur.] → {len(asignadas)} entidades, "
              f"{po[p]:.0f} pts")
        # imprimir en líneas de ~60 caracteres
        linea = "    "
        for n, w in asignadas:
            tok = f"{n} ({w})  "
            if len(linea) + len(tok) > 70:
                print(linea)
                linea = "    "
            linea += tok
        if linea.strip():
            print(linea)

    print("\n" + "="*62)

In [41]:
# ─────────────────────────────────────────────────────
# 8. PUNTO DE ENTRADA
# ─────────────────────────────────────────────────────

if __name__ == "__main__":

    print("="*62)
    print("  VERDADERA DEMOCRACIA — Algoritmo Genético")
    print("="*62)

    # Generar datos
    curules, pesos = generar_datos_aleatorios()

    print("\n  CURULES POR PARTIDO:")
    for i, (p, c) in enumerate(zip(PARTIDOS, curules)):
        barra = "█" * c
        print(f"  {p:<12} {c:>3} curules  {barra}")

    print(f"\n  PESOS DE ENTIDADES (muestra):")
    for i in range(10):
        print(f"    {i+1:>2}. {ENTIDADES[i]:<25} {pesos[i]:>3} pts")
    print(f"    ... (40 entidades más)")

    # Ejecutar AG
    print("\n  EJECUTANDO ALGORITMO GENÉTICO...\n")
    mejor, fit, historial = algoritmo_genetico(
        curules         = curules,
        pesos           = pesos,
        tam_poblacion   = 200,
        n_generaciones  = 10000,
        prob_cruce      = 0.80,
        prob_mutacion   = 0.04,
        objetivo_fitness= 0.98,
        verbose         = True,
    )

    # Mostrar resultados
    mostrar_resultados(mejor, curules, pesos, fit)

    # Exportar cromosoma como dict legible
    print("\n  CROMOSOMA FINAL (entidad → partido):")
    asignacion = {ENTIDADES[i]: PARTIDOS[mejor[i]] for i in range(N_ENTIDADES)}
    for entidad, partido in asignacion.items():
        print(f"    {entidad:<25} → {partido}")


  VERDADERA DEMOCRACIA — Algoritmo Genético

  CURULES POR PARTIDO:
  Partido A      9 curules  █████████
  Partido B     12 curules  ████████████
  Partido C     27 curules  ███████████████████████████
  Partido D      1 curules  █
  Partido E      1 curules  █

  PESOS DE ENTIDADES (muestra):
     1. Min. Hacienda              29 pts
     2. Min. Interior              49 pts
     3. Min. Defensa               75 pts
     4. Min. Relaciones Ext.       18 pts
     5. Min. Justicia              67 pts
     6. Min. Educación             83 pts
     7. Min. Salud                 38 pts
     8. Min. Trabajo               21 pts
     9. Min. Comercio              36 pts
    10. Min. Ambiente              61 pts
    ... (40 entidades más)

  EJECUTANDO ALGORITMO GENÉTICO...

  Gen    0 | fitness = 0.000820
  Gen  100 | fitness = 0.085324
  Gen  200 | fitness = 0.151515
  Gen  300 | fitness = 0.217391
  Gen  400 | fitness = 0.217391
  Gen  500 | fitness = 0.217391
  Gen  600 | fitness = 0.352

# Análisis

## Estancamiento del Fitness y Diversidad Genética

El algoritmo genético ejecutado alcanzó un `fitness` de **0.471698** después de 10000 generaciones, sin lograr el objetivo de `0.98`. Este estancamiento es un fenómeno común conocido como **convergencia prematura**, donde el algoritmo deja de encontrar soluciones mejores a pesar de continuar las iteraciones.

Las razones principales para esto suelen ser:

*   **Pérdida de Diversidad Genética**: La población se vuelve demasiado homogénea, lo que limita la capacidad de los operadores genéticos (cruce y mutación) para generar nuevas soluciones prometedoras. Todos los individuos comienzan a parecerse al 'mejor' encontrado hasta ese momento.
*   **Atrapamiento en Óptimos Locales**: El algoritmo puede haber encontrado un buen sub-óptimo en el 'paisaje' de soluciones, pero no logra 'saltar' a una región donde podría existir una solución globalmente mejor.

### Impacto de las Tasas de Mutación y Cruce

Las tasas de mutación y cruce juegan un papel crucial:

*   **`prob_mutacion` (0.04)**: Una tasa de mutación *demasiado baja* puede no introducir suficiente variabilidad para explorar nuevas áreas del espacio de búsqueda o escapar de óptimos locales. Una tasa *demasiado alta* puede destruir soluciones prometedoras.
*   **`prob_cruce` (0.80)**: Una tasa de cruce alta favorece la recombinación de material genético. Si la diversidad ya es baja, el cruce entre individuos similares no aportará mucha novedad.

En este caso, a pesar de 10,000 generaciones, el fitness solo llegó a 0.47, lo que sugiere que la combinación actual de parámetros (`tam_poblacion`, `prob_cruce`, `prob_mutacion`) y quizás el tamaño de la `poblacion` no fueron suficientes para superar el estancamiento o explorar el espacio de soluciones de manera más efectiva para este problema específico. El algoritmo convergió a una solución que es buena, pero no la 'casi perfecta' deseada por el `objetivo_fitness`.